In [1]:
import os

In [2]:
%pwd

'd:\\End_to_End_MLOps_project_for_Receipt_OCR_and_Document_Intelligent\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\End_to_End_MLOps_project_for_Receipt_OCR_and_Document_Intelligent'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from src.receipt_intelligence.constants import *
from src.receipt_intelligence.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [15]:
import logging
import shutil
import zipfile
from pathlib import Path
from urllib.parse import parse_qs, urlparse

import gdown


logger = logging.getLogger(__name__)


class DataIngestion:
    """
    Production-grade data ingestion pipeline for a Google Drive ZIP file.

    Pipeline:

        Google Drive file
              ↓
        Download ZIP
              ↓
        Validate ZIP
              ↓
        Secure extraction
              ↓
        Return extracted dataset directory

    Expected config attributes:
        source_URL
        local_data_file
        unzip_dir
    """

    def __init__(self, config):
        self.config = config

        self.source_url = str(config.source_URL).strip()
        self.download_path = Path(config.local_data_file)
        self.extract_path = Path(config.unzip_dir)

        if not self.source_url:
            raise ValueError("Google Drive source_URL cannot be empty.")

        if not self.download_path:
            raise ValueError("local_data_file cannot be empty.")

        if not self.extract_path:
            raise ValueError("unzip_dir cannot be empty.")

    # ==========================================================
    # GOOGLE DRIVE URL VALIDATION
    # ==========================================================

    @staticmethod
    def _extract_google_drive_file_id(url: str) -> str:
        """
        Extract Google Drive file ID from common Google Drive URLs.

        Supported examples:

            https://drive.google.com/file/d/<FILE_ID>/view

            https://drive.google.com/uc?id=<FILE_ID>

            https://drive.google.com/open?id=<FILE_ID>
        """

        parsed = urlparse(url)

        # ------------------------------------------------------
        # /file/d/<ID>/view
        # ------------------------------------------------------
        if "/file/d/" in parsed.path:

            parts = parsed.path.split("/file/d/")

            if len(parts) == 2:

                file_id = parts[1].split("/")[0].strip()

                if file_id:
                    return file_id

        # ------------------------------------------------------
        # ?id=<ID>
        # ------------------------------------------------------
        query_params = parse_qs(parsed.query)

        file_id = query_params.get("id")

        if file_id and file_id[0].strip():
            return file_id[0].strip()

        raise ValueError(
            f"Unable to extract Google Drive file ID from URL: {url}"
        )

    # ==========================================================
    # DOWNLOAD GOOGLE DRIVE ZIP FILE
    # ==========================================================

    def download_file(self) -> str:
        """
        Download a ZIP file from Google Drive.

        Returns:
            str:
                Path to the downloaded ZIP file.
        """

        try:

            logger.info(
                "Starting Google Drive file download..."
            )

            logger.info(
                "Source URL: %s",
                self.source_url
            )

            logger.info(
                "Download path: %s",
                self.download_path
            )

            # --------------------------------------------------
            # Create parent directory
            # --------------------------------------------------

            self.download_path.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # --------------------------------------------------
            # Extract Google Drive file ID
            # --------------------------------------------------

            file_id = self._extract_google_drive_file_id(
                self.source_url
            )

            logger.info(
                "Google Drive file ID detected: %s",
                file_id
            )

            # --------------------------------------------------
            # Convert to direct download URL
            # --------------------------------------------------

            download_url = (
                f"https://drive.google.com/uc?id={file_id}"
            )

            logger.info(
                "Using Google Drive direct download endpoint."
            )

            # --------------------------------------------------
            # Remove stale file if it exists
            # --------------------------------------------------

            if self.download_path.exists():

                logger.warning(
                    "Existing download found. Removing: %s",
                    self.download_path
                )

                if self.download_path.is_dir():
                    shutil.rmtree(self.download_path)

                else:
                    self.download_path.unlink()

            # --------------------------------------------------
            # Download
            # --------------------------------------------------

            downloaded_path = gdown.download(
                url=download_url,
                output=str(self.download_path),
                quiet=False
            )

            if downloaded_path is None:
                raise RuntimeError(
                    "gdown failed to download the Google Drive file."
                )

            downloaded_path = Path(downloaded_path)

            # --------------------------------------------------
            # Validate file exists
            # --------------------------------------------------

            if not downloaded_path.exists():

                raise FileNotFoundError(
                    "Downloaded file does not exist: "
                    f"{downloaded_path}"
                )

            if not downloaded_path.is_file():

                raise RuntimeError(
                    "Downloaded path is not a regular file: "
                    f"{downloaded_path}"
                )

            # --------------------------------------------------
            # Validate file size
            # --------------------------------------------------

            file_size = downloaded_path.stat().st_size

            if file_size <= 0:

                raise RuntimeError(
                    "Downloaded file is empty."
                )

            logger.info(
                "Downloaded file size: %.2f MB",
                file_size / (1024 * 1024)
            )

            # --------------------------------------------------
            # Validate ZIP
            # --------------------------------------------------

            if not zipfile.is_zipfile(downloaded_path):

                # Read first bytes to detect common HTML errors.
                try:

                    with open(
                        downloaded_path,
                        "rb"
                    ) as f:

                        header = f.read(500)

                except Exception:

                    header = b""

                header_lower = header.lower()

                if (
                    b"<html" in header_lower
                    or b"<!doctype" in header_lower
                    or b"google drive" in header_lower
                ):

                    raise RuntimeError(
                        "Google Drive returned an HTML page instead "
                        "of the expected ZIP file. "
                        "Check that the file is shared as "
                        "'Anyone with the link'."
                    )

                raise RuntimeError(
                    f"Downloaded file is not a valid ZIP archive: "
                    f"{downloaded_path}"
                )

            logger.info(
                "ZIP validation successful."
            )

            logger.info(
                "Google Drive file downloaded successfully: %s",
                downloaded_path
            )

            return str(downloaded_path)

        except Exception as e:

            logger.exception(
                "Failed to download Google Drive ZIP file."
            )

            raise RuntimeError(
                "Google Drive ZIP download failed."
            ) from e

    # ==========================================================
    # SECURE ZIP EXTRACTION
    # ==========================================================

    @staticmethod
    def _validate_zip_members(
        zip_ref: zipfile.ZipFile,
        extraction_path: Path
    ) -> None:
        """
        Validate ZIP members against path traversal attacks.
        """

        extraction_root = extraction_path.resolve()

        for member in zip_ref.infolist():

            member_path = (
                extraction_root / member.filename
            ).resolve()

            try:

                member_path.relative_to(
                    extraction_root
                )

            except ValueError:

                raise RuntimeError(
                    "Unsafe ZIP archive detected. "
                    f"Path traversal attempt: {member.filename}"
                )

    # ==========================================================
    # EXTRACT ZIP FILE
    # ==========================================================

    def extract_zip_file(self) -> str:
        """
        Extract the downloaded ZIP file securely.

        Returns:
            str:
                Extraction directory.
        """

        try:

            zip_path = Path(
                self.download_path
            )

            # --------------------------------------------------
            # Validate ZIP exists
            # --------------------------------------------------

            if not zip_path.exists():

                raise FileNotFoundError(
                    f"ZIP file not found: {zip_path}"
                )

            if not zip_path.is_file():

                raise RuntimeError(
                    f"ZIP path is not a file: {zip_path}"
                )

            # --------------------------------------------------
            # Validate ZIP format
            # --------------------------------------------------

            if not zipfile.is_zipfile(zip_path):

                raise RuntimeError(
                    f"Invalid ZIP archive: {zip_path}"
                )

            # --------------------------------------------------
            # Create extraction directory
            # --------------------------------------------------

            self.extract_path.mkdir(
                parents=True,
                exist_ok=True
            )

            logger.info(
                "Extraction directory: %s",
                self.extract_path
            )

            # --------------------------------------------------
            # Open ZIP
            # --------------------------------------------------

            with zipfile.ZipFile(
                zip_path,
                mode="r"
            ) as zip_ref:

                members = zip_ref.infolist()

                if not members:

                    raise RuntimeError(
                        "ZIP archive is empty."
                    )

                logger.info(
                    "ZIP contains %d entries.",
                    len(members)
                )

                # --------------------------------------------------
                # Security validation
                # --------------------------------------------------

                self._validate_zip_members(
                    zip_ref,
                    self.extract_path
                )

                # --------------------------------------------------
                # Extract
                # --------------------------------------------------

                for member in members:

                    logger.debug(
                        "Extracting: %s",
                        member.filename
                    )

                    zip_ref.extract(
                        member,
                        path=self.extract_path
                    )

            # --------------------------------------------------
            # Validate extraction
            # --------------------------------------------------

            extracted_files = [
                p
                for p in self.extract_path.rglob("*")
                if p.is_file()
            ]

            if not extracted_files:

                raise RuntimeError(
                    "ZIP extraction completed but no files "
                    "were found."
                )

            logger.info(
                "Successfully extracted %d files.",
                len(extracted_files)
            )

            for file_path in extracted_files:

                logger.debug(
                    "Extracted: %s",
                    file_path
                )

            logger.info(
                "ZIP extraction completed successfully."
            )

            return str(self.extract_path)

        except Exception as e:

            logger.exception(
                "Failed to extract ZIP file."
            )

            raise RuntimeError(
                "ZIP extraction failed."
            ) from e

    # ==========================================================
    # COMPLETE INGESTION PIPELINE
    # ==========================================================

    def run(self) -> str:
        """
        Execute complete data ingestion pipeline.

        Pipeline:

            Google Drive ZIP
                    ↓
            Download
                    ↓
            Validate
                    ↓
            Secure extraction
                    ↓
            Return extracted directory

        Returns:
            str:
                Path to extracted dataset.
        """

        logger.info(
            "Starting data ingestion..."
        )

        # ------------------------------------------------------
        # Step 1: Download
        # ------------------------------------------------------

        zip_path = self.download_file()

        logger.info(
            "Download completed: %s",
            zip_path
        )

        # ------------------------------------------------------
        # Step 2: Extract
        # ------------------------------------------------------

        extracted_path = self.extract_zip_file()

        logger.info(
            "Extraction completed: %s",
            extracted_path
        )

        # ------------------------------------------------------
        # Final validation
        # ------------------------------------------------------

        extracted_files = [
            p
            for p in Path(extracted_path).rglob("*")
            if p.is_file()
        ]

        if not extracted_files:

            raise RuntimeError(
                "Data ingestion completed but the extracted "
                "directory contains no files."
            )

        logger.info(
            "Final dataset validation successful."
        )

        logger.info(
            "Total extracted files: %d",
            len(extracted_files)
        )

        logger.info(
            "Data ingestion completed successfully."
        )

        return str(extracted_path)

In [16]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.run()
except Exception as e:
    raise e

[2026-08-30 18:44:04,283: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-30 18:44:04,289: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-30 18:44:04,292: INFO: common: created directory at: artifacts]
[2026-08-30 18:44:04,297: INFO: common: created directory at: artifacts/data_ingestion]
[2026-08-30 18:44:04,298: INFO: 260946554: Starting data ingestion...]
[2026-08-30 18:44:04,300: INFO: 260946554: Starting Google Drive file download...]
[2026-08-30 18:44:04,304: INFO: 260946554: Source URL: https://drive.google.com/file/d/13GRmxB4JS2d-5MYqWV63zYXRO5zhEjYp/view]
[2026-08-30 18:44:04,306: INFO: 260946554: Download path: artifacts\data_ingestion\data.zip]
[2026-08-30 18:44:04,310: INFO: 260946554: Google Drive file ID detected: 13GRmxB4JS2d-5MYqWV63zYXRO5zhEjYp]
[2026-08-30 18:44:04,314: INFO: 260946554: Using Google Drive direct download endpoint.]


Downloading...
From (original): https://drive.google.com/uc?id=13GRmxB4JS2d-5MYqWV63zYXRO5zhEjYp
From (redirected): https://drive.google.com/uc?id=13GRmxB4JS2d-5MYqWV63zYXRO5zhEjYp&confirm=t&uuid=527e8019-a03f-4118-804b-b0641b83e7bd
To: d:\End_to_End_MLOps_project_for_Receipt_OCR_and_Document_Intelligent\artifacts\data_ingestion\data.zip
100%|██████████| 148M/148M [00:27<00:00, 5.37MB/s] 

[2026-08-30 18:44:35,547: INFO: 260946554: Downloaded file size: 141.50 MB]
[2026-08-30 18:44:35,631: INFO: 260946554: ZIP validation successful.]
[2026-08-30 18:44:35,638: INFO: 260946554: Google Drive file downloaded successfully: artifacts\data_ingestion\data.zip]
[2026-08-30 18:44:35,638: INFO: 260946554: Download completed: artifacts\data_ingestion\data.zip]
[2026-08-30 18:44:35,646: INFO: 260946554: Extraction directory: artifacts\data_ingestion]


[2026-08-30 18:44:35,654: INFO: 260946554: ZIP contains 371 entries.]
[2026-08-30 18:44:38,029: INFO: 260946554: Successfully extracted 372 files.]
[2026-08-30 18:44:38,029: INFO: 260946554: ZIP extraction completed successfully.]
[2026-08-30 18:44:38,029: INFO: 260946554: Extraction completed: artifacts\data_ingestion]
[2026-08-30 18:44:38,173: INFO: 260946554: Final dataset validation successful.]
[2026-08-30 18:44:38,173: INFO: 260946554: Total extracted files: 372]
[2026-08-30 18:44:38,188: INFO: 260946554: Data ingestion completed successfully.]
